# ML Dataset Preparation

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

np.random.seed(42)

n = 1000

df = pd.DataFrame({
    "customer_id": range(1001, 1001 + n),
    "age": np.random.randint(18, 71, n),
    "income": np.random.normal(55000, 18000, n).round(2),
    "account_tenure_months": np.random.randint(1, 121, n),
    "monthly_spend": np.random.gamma(4, 800, n).round(2),
    "support_tickets": np.random.poisson(2, n),
    "membership": np.random.choice(
        ["Basic", "Silver", "Gold"],
        n,
        p=[0.50, 0.30, 0.20]
    ),
    "city": np.random.choice(
        ["Delhi", "Mumbai", "Pune", "Jaipur"],
        n
    ),
    "marketing_channel": np.random.choice(
        ["Email", "Social", "Paid Search", "Organic"],
        n
    ),
    "churn": np.random.choice([0, 1], n, p=[0.75, 0.25])
})

df.head()

,customer_id,age,income,account_tenure_months,monthly_spend,support_tickets,membership,city,marketing_channel,churn
0,1001,56,69394.44,108,2554.48,1,Basic,Jaipur,Organic,1
1,1002,69,56242.10,5,6127.01,3,Basic,Mumbai,Organic,0
2,1003,46,68837.09,3,4135.76,2,Silver,Jaipur,Social,1
3,1004,32,40953.82,65,2451.10,3,Gold,Mumbai,Organic,0
4,1005,60,87053.13,4,2772.01,2,Basic,Pune,Paid Search,0


#### Q1. Inspect the ML dataset
Perform a complete initial inspection of df.
- Display its shape and data types.
- Check missing values in every column.
- Check duplicate rows and duplicate customer_id values.
- Display summary statistics for numerical columns.
- Calculate the distribution of the target churn.

In [10]:
display('shape: ',df.shape,'data types', df.dtypes)

'shape: '

(1000, 10)

'data types'

customer_id                int64
age                        int32
income                   float64
account_tenure_months      int32
monthly_spend            float64
support_tickets            int32
membership                object
city                      object
marketing_channel         object
churn                      int32
dtype: object

In [16]:
df[df.duplicated(keep='last')]

,customer_id,age,income,account_tenure_months,monthly_spend,support_tickets,membership,city,marketing_channel,churn


In [18]:
df[df['customer_id'].duplicated(keep=False)]

,customer_id,age,income,account_tenure_months,monthly_spend,support_tickets,membership,city,marketing_channel,churn


In [20]:
df.isnull().sum()

customer_id              0
age                      0
income                   0
account_tenure_months    0
monthly_spend            0
support_tickets          0
membership               0
city                     0
marketing_channel        0
churn                    0
dtype: int64

In [22]:
df.describe()

,customer_id,age,income,account_tenure_months,monthly_spend,support_tickets,churn
count,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,1500.500000,44.385000,55298.968390,59.919000,3251.886860,1.929000,0.251000
std,288.819436,15.321669,18950.211244,34.441583,1631.372549,1.311365,0.433805
min,1001.000000,18.000000,-9227.700000,1.000000,313.120000,0.000000,0.000000
25%,1250.750000,31.750000,43318.172500,30.000000,2070.050000,1.000000,0.000000
50%,1500.500000,45.000000,55894.240000,59.000000,3007.305000,2.000000,0.000000
75%,1750.250000,57.000000,68326.005000,89.250000,4082.480000,3.000000,1.000000
max,2000.000000,70.000000,112139.960000,120.000000,13995.160000,8.000000,1.000000


In [24]:
df['churn'].value_counts(normalize=True)*100

churn
0    74.9
1    25.1
Name: proportion, dtype: float64

#### Q2. Separate features and target
Create:
- X: all predictor columns.
- y: the target column churn.
Verify their shapes and confirm that churn is not present in X.

In [29]:
y=df['churn']
X=df.drop(columns='churn')
X.columns

Index(['customer_id', 'age', 'income', 'account_tenure_months',
       'monthly_spend', 'support_tickets', 'membership', 'city',
       'marketing_channel'],
      dtype='object')

In [33]:
display(X.shape,y.shape)

(1000, 9)

(1000,)

#### Q3. Identify feature types
From X, create two lists:
- numeric_features: numerical predictor columns.
- categorical_features: categorical predictor columns.
Use Pandas dtypes rather than manually typing the column names.

Exclude customer_id from predictive features and explain why identifiers are not automatically useful predictors.

In [36]:
numeric_features=X.select_dtypes(include=['float','int']).columns.tolist()

In [38]:
numeric_features

['customer_id',
 'age',
 'income',
 'account_tenure_months',
 'monthly_spend',
 'support_tickets']

In [40]:
categorical_features=X.select_dtypes(include=['object']).columns.tolist()
categorical_features

['membership', 'city', 'marketing_channel']

#### Q4. Perform a basic train-test split
Split X and y into training and testing datasets using:
- test_size=0.20
- random_state=42
Display the shapes of all four resulting objects.

In [81]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)
display('complete dataset',df['churn'].value_counts(normalize=True)*100)
display('training dataset',y_train.value_counts(normalize=True)*100)
display('complete dataset',y_test.value_counts(normalize=True)*100)

'complete dataset'

churn
0    74.9
1    25.1
Name: proportion, dtype: float64

'training dataset'

churn
0    75.0
1    25.0
Name: proportion, dtype: float64

'complete dataset'

churn
0    74.5
1    25.5
Name: proportion, dtype: float64

In [52]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

In [54]:
display(X_train.shape,y_train.shape,X_test.shape,y_test.shape)

(800, 9)

(800,)

(200, 9)

(200,)

#### Q5. Verify the split
Check that:
- Training and testing indices do not overlap.
- Their combined row counts equal the original dataset size.
- No customer IDs occur in both splits.
For this dataset, use the original customer_id column to verify record separation before excluding it from X.

In [56]:
print("Overlapping indices:",
      len(X_train.index.intersection(X_test.index)))

Overlapping indices: 0


In [60]:
print("Original dataset rows:", len(df))
print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Combined rows:", len(X_train) + len(X_test))

assert len(X_train) + len(X_test) == len(df), "Row counts do not match!"

print("Verified: Combined row counts equal the original dataset size.")

Original dataset rows: 1000
Training rows: 800
Testing rows: 200
Combined rows: 1000
Verified: Combined row counts equal the original dataset size.


#### Q6. Compare target distributions
Calculate the proportion of churned and non-churned customers in:
- The complete dataset.
- The training set.
- The testing set.

In [73]:
display('complete dataset',df['churn'].value_counts(normalize=True)*100)


'complete dataset'

churn
0    74.9
1    25.1
Name: proportion, dtype: float64

In [77]:
display('training dataset',y_train.value_counts(normalize=True)*100)
display('complete dataset',y_test.value_counts(normalize=True)*100)

'training dataset'

churn
0    74.875
1    25.125
Name: proportion, dtype: float64

'complete dataset'

churn
0    75.0
1    25.0
Name: proportion, dtype: float64

#### Q8. Compare different test sizes
Perform splits with test_size=0.10, 0.20, and 0.30, keeping random_state=42 and using stratify=y.

Record the training and testing row counts for each split.

In [88]:
ts=[0.1,0.2,0.3]
for i in ts:
    X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=i,random_state=42,stratify=y)
    print("split size ",i)
    display(len(X_train),len(X_test))



split size  0.1


900

100

split size  0.2


800

200

split size  0.3


700

300

#### Q9. Detect target leakage
Create a temporary column that directly copies the target:
df["churn_leakage"] = df["churn"]



Demonstrate how accidentally including this column in X can make a model's evaluation misleading.

Then remove it and explain why features must not reveal the target directly or indirectly.